# CampusHire: Student Placement Eligibility Predictor
### Binary Classification using Logistic Regression
**Problem 07** from NoviTech Applied Machine Learning Practice
---
### Business Context & Objective
University placement cells must efficiently screen student profiles against company eligibility requirements. The objective of this project is to build an interpretable binary classification model using **Logistic Regression** to predict whether a candidate is likely to meet placement eligibility criteria (`target = 1`) or not (`target = 0`).

### Predictor Features:
- `cgpa`: Cumulative Grade Point Average (scale 4.0 to 10.0)
- `attendance_pct`: Course attendance percentage (0 to 100%)
- `coding_score`: Technical assessment rating (scale 0 to 10)
- `projects_completed`: Number of software/engineering projects built
- `internship_months`: Total duration of industry internships completed (months)
- `backlogs`: Number of uncleared backlogs

## Step 1: Import Libraries and Load Dataset
We use standard data science libraries: `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scikit-learn`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Load dataset
df = pd.read_csv('data/student_placement_eligibility.csv')
print('Dataset Shape:', df.shape)
df.head()

## Step 2: Exploratory Data Analysis (EDA)
Checking missing values, duplicate entries, and class distribution.

In [ ]:
print('Missing Values Check:')
print(df.isnull().sum())
print('\nDuplicates Check:', df.duplicated().sum())
print('\nTarget Class Balance:')
print(df['target'].value_counts(normalize=True))

## Step 3: Train / Test Split (Without Data Leakage)
We split the dataset into 80% train and 20% test with stratified sampling on `target` to maintain the 50/50 balance.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

## Step 4: Feature Scaling (StandardScaler)
Because features have different units (e.g. CGPA is 4-10 while attendance is 0-100), we apply `StandardScaler`. We fit exclusively on `X_train` to prevent data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('StandardScaler applied successfully.')

## Step 5: Establish Baseline Model
A simple dummy classifier (always predicting most frequent class) provides a 50% benchmark.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print(f'Baseline Accuracy: {baseline.score(X_test, y_test):.4f} (50% random benchmark)')

## Step 6: Train Logistic Regression Model

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)
print('Logistic Regression Model trained successfully.')

## Step 7: Model Evaluation & Confusion Matrix
Evaluating test set predictions with Accuracy, Precision, Recall, F1, and ROC-AUC.

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Not Eligible (0)', 'Eligible (1)'],
            yticklabels=['Not Eligible (0)', 'Eligible (1)'])
plt.title('Confusion Matrix - CampusHire Eligibility Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Step 8: Coefficient & Feature Impact Interpretation
Positive coefficients increase the odds of eligibility, whereas negative coefficients decrease the odds.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0],
    'Odds_Ratio': np.exp(model.coef_[0])
}).sort_values(by='Coefficient', ascending=True)

print(coef_df)

plt.figure(figsize=(8, 4))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linestyle='--')
plt.title('Feature Impact on Placement Eligibility (Log-Odds)')
plt.xlabel('Coefficient')
plt.show()

## Step 9: Make Predictions on New Students

In [ ]:
def predict_student(cgpa, attendance_pct, coding_score, projects_completed, internship_months, backlogs):
    new_data = pd.DataFrame([{
        'cgpa': cgpa, 'attendance_pct': attendance_pct, 'coding_score': coding_score,
        'projects_completed': projects_completed, 'internship_months': internship_months, 'backlogs': backlogs
    }])
    new_scaled = scaler.transform(new_data)
    prob = model.predict_proba(new_scaled)[0, 1]
    decision = 'ELIGIBLE FOR PLACEMENT' if prob >= 0.5 else 'NOT ELIGIBLE'
    return decision, prob

# Test Student Profile
decision, prob = predict_student(cgpa=7.8, attendance_pct=85.0, coding_score=3, projects_completed=5, internship_months=12, backlogs=4)
print(f'Prediction: {decision} with {prob*100:.1f}% probability of eligibility.')